# AI Risk Manager - Data Exploration
## Triple Threat Chargeback Defense System

This notebook explores the transaction dataset to understand fraud patterns, payment methods, and merchant categories.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data.schema import FraudLabel, PaymentMethod, MerchantCategory

## Load Data
Load the generated transaction data and inspect its structure.

In [ ]:
df = pd.read_csv("../data/transactions.csv")
print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nFirst 5 rows:")
df.head()

## Basic Statistics
High-level summary of the dataset including date range, amount range, and fraud rate.

In [ ]:
print("=== Dataset Summary ===")
print(f"Total transactions: {len(df)}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Amount range: ₹{df['amount'].min():,.0f} to ₹{df['amount'].max():,.0f}")
print(f"\nChargeback rate: {df['chargeback_label'].mean():.1%}")

## Fraud Distribution
Visual breakdown of fraud types and chargeback label distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fraud_counts = df['fraud_type'].value_counts()
axes[0].pie(fraud_counts.values, labels=fraud_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Fraud Type Distribution')

sns.countplot(data=df, x='chargeback_label', ax=axes[1])
axes[1].set_title('Chargeback Label Distribution')
axes[1].set_xticklabels(['Legitimate', 'Fraud'])

plt.tight_layout()
plt.savefig('../evaluation/reports/fraud_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Amount Analysis
Distribution of transaction amounts across different fraud types.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for fraud_type in df['fraud_type'].unique():
    subset = df[df['fraud_type'] == fraud_type]
    axes[0].hist(subset['amount'], alpha=0.5, label=fraud_type, bins=30)
axes[0].set_title('Amount Distribution by Fraud Type')
axes[0].set_xlabel('Amount (INR)')
axes[0].legend()

sns.boxplot(data=df, x='fraud_type', y='amount', ax=axes[1])
axes[1].set_title('Amount by Fraud Type')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../evaluation/reports/amount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Payment Methods
Analysis of payment method usage and associated fraud rates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

payment_counts = df['payment_method'].value_counts()
axes[0].barh(payment_counts.index, payment_counts.values)
axes[0].set_title('Payment Method Distribution')

fraud_by_payment = df.groupby('payment_method')['chargeback_label'].mean().sort_values(ascending=False)
axes[1].barh(fraud_by_payment.index, fraud_by_payment.values)
axes[1].set_title('Fraud Rate by Payment Method')
axes[1].set_xlabel('Fraud Rate')

plt.tight_layout()
plt.savefig('../evaluation/reports/payment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Merchant Categories
Distribution and fraud rates across merchant categories.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

merchant_counts = df['merchant_category'].value_counts()
axes[0].barh(merchant_counts.index, merchant_counts.values)
axes[0].set_title('Merchant Category Distribution')

fraud_by_merchant = df.groupby('merchant_category')['chargeback_label'].mean().sort_values(ascending=False)
axes[1].barh(fraud_by_merchant.index, fraud_by_merchant.values)
axes[1].set_title('Fraud Rate by Merchant Category')
axes[1].set_xlabel('Fraud Rate')

plt.tight_layout()
plt.savefig('../evaluation/reports/merchant_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Correlations
Correlation of numeric features with the chargeback label.

In [ ]:
numeric_cols = ['amount', 'account_age_days', 'past_disputes', 'is_new_device', 'is_new_address']
corr_with_target = df[numeric_cols + ['chargeback_label']].corr()['chargeback_label'].drop('chargeback_label')

plt.figure(figsize=(10, 6))
corr_with_target.sort_values().plot(kind='barh')
plt.title('Feature Correlations with Chargeback')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.savefig('../evaluation/reports/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## Temporal Patterns
Fraud patterns across hours of the day and days of the week.

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fraud_by_hour = df.groupby('hour')['chargeback_label'].mean()
axes[0].plot(fraud_by_hour.index, fraud_by_hour.values, marker='o')
axes[0].set_title('Fraud Rate by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Fraud Rate')
axes[0].grid(True, alpha=0.3)

fraud_by_day = df.groupby('day_of_week')['chargeback_label'].mean()
axes[1].bar(fraud_by_day.index, fraud_by_day.values)
axes[1].set_title('Fraud Rate by Day of Week')
axes[1].set_xlabel('Day (0=Monday, 6=Sunday)')
axes[1].set_ylabel('Fraud Rate')

plt.tight_layout()
plt.savefig('../evaluation/reports/temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary
Key findings from the data exploration.

In [ ]:
print("=== Key Findings ===")
print(f"1. Dataset contains {len(df)} transactions with {df['chargeback_label'].mean():.1%} fraud rate")
print(f"2. Most common fraud type: {df['fraud_type'].mode()[0]}")
print(f"3. Average transaction amount: ₹{df['amount'].mean():,.0f}")
print(f"4. Most used payment method: {df['payment_method'].mode()[0]}")
print(f"5. Highest fraud category: {df.groupby('merchant_category')['chargeback_label'].mean().idxmax()}")